# 01 · Ingest sample AMR CSV → Lakehouse table `isolates`

In [ ]:
# Detect path for Fabric vs local dev
import os
from pathlib import Path

candidate_paths = [
    "Files/fabric/data/sample/sample_amr.csv",   # Microsoft Fabric (OneLake Files)
    "./fabric/data/sample/sample_amr.csv",       # Local repo relative
    "fabric/data/sample/sample_amr.csv"          # Local repo alternate
]
read_path = next((p for p in candidate_paths if Path(p).exists()), candidate_paths[0])
print(f"Using input: {read_path}")

In [ ]:
from pyspark.sql import functions as F, types as T

# Read CSV with header + schema inference
df_raw = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(read_path)
)
print("Raw rows:", df_raw.count())
df_raw.printSchema()

In [ ]:
# Normalize columns & types expected by downstream notebooks
required_cols = [
    "province", "organism", "antibiotic", "specimen", "sector",
    "year", "n_tested", "percent_resistant"
]

missing = [c for c in required_cols if c not in df_raw.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = (df_raw
    .select(*required_cols)
    .withColumn("year", F.col("year").cast("int"))
    .withColumn("n_tested", F.col("n_tested").cast("int"))
    .withColumn("percent_resistant", F.col("percent_resistant").cast("double"))
)

# Basic sanity: no null keys, sensible ranges
problems = df.where(
    F.col("province").isNull() | F.col("organism").isNull() | F.col("antibiotic").isNull() |
    F.col("specimen").isNull() | F.col("sector").isNull() |
    (F.col("year") < 2000) | (F.col("year") > 2100) |
    (F.col("n_tested") <= 0) |
    (F.col("percent_resistant") < 0) | (F.col("percent_resistant") > 100)
).limit(5)

if problems.count() > 0:
    display(problems)
    raise ValueError("Data validation failed; see rows above")

display(df.limit(10))

In [ ]:
# Write managed Delta table in the Lakehouse catalog: `isolates`
spark.sql("DROP TABLE IF EXISTS isolates")
(
  df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("isolates")
)
print("Wrote Delta table: isolates")
spark.sql("REFRESH TABLE isolates")
display(spark.table("isolates").orderBy("province", "organism", "antibiotic", "year"))